# 02g Final m9_pbm Evaluation

**Research question.** How well does the final deterministic physical procedure
classify days, localise correction windows, recover correction energy, and
reduce manual review on held-out Beta substations?

This notebook uses only the nested outer predictions from 02e. Each Beta
substation-day was predicted by weights and a threshold selected without that
substation's labels. The notebook expands positive windows to 15-minute flags
and reports Beta sure as primary, with Beta all as a sensitivity analysis.

**Inputs:** final Beta data and 02e nested outer predictions.  
**Outputs:** five metric CSVs, four compact tables, five figures, a local
interval audit, and a manifest.  
**Expected runtime:** under five minutes.

## 1. Imports, Paths, And Operating-Point Configuration

Confidence coverage levels and the development operating-point constraints are
read from configuration. The recommended point is the largest tested coverage
with auto-accepted precision at least 0.99 and F1 at least 0.95; if no tested
level qualifies, that absence is reported explicitly.

In [ ]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display


def find_notebook_directory() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if candidate.name == "notebooks" and (candidate / "_m9_pbm_data.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article" / "notebooks"
        if (nested / "_m9_pbm_data.py").exists():
            return nested
    raise FileNotFoundError("Could not locate the journal notebook directory.")


NOTEBOOK_DIR = find_notebook_directory()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _m9_pbm_data import (  # noqa: E402
    load_dataset,
    load_experiment_config,
    manifest_payload,
    output_dirs,
    resolve_paths,
    write_csv,
    write_manifest,
    write_parquet,
)
from _m9_pbm_features import window_iou  # noqa: E402
from _m9_pbm_plotting import (  # noqa: E402
    plot_auto_accept_burden,
    plot_coverage_scores,
    plot_energy_summary,
    plot_final_confusion_matrices,
    plot_window_iou_distribution,
)
from _m9_pbm_validation import (  # noqa: E402
    confidence_coverage_metrics,
    metric_rows,
    recommended_coverage,
)

STARTED_AT = time.time()
ARTICLE_ROOT = NOTEBOOK_DIR.parent
CONFIG = load_experiment_config(ARTICLE_ROOT)
PATHS = resolve_paths(ARTICLE_ROOT, CONFIG)
SLUG = "02g_m9_pbm_final_evaluation"
OUTPUT_DIRS = output_dirs(PATHS, SLUG)
COVERAGE_CONFIG = CONFIG["m9_pbm"]["confidence_coverage"]

display(pd.Series(COVERAGE_CONFIG, name="confidence_coverage"))

## 2. Build The Held-Out Interval Audit

Every positive day uses the candidate selected by its outer-fold weight vector.
For a negative day, no interval is predicted positive even though the day still
has a highest-scoring candidate. The join must preserve all 280,800 final Beta
rows and exactly one daily prediction key.

In [ ]:
PREDICTION_PATH = (
    PATHS.intermediate
    / "02e_m9_pbm_weight_optimisation"
    / "nested_outer_predictions.parquet"
)
assert PREDICTION_PATH.exists(), "Run Notebook 02e first."
predictions = pd.read_parquet(PREDICTION_PATH)
beta = load_dataset("beta", article_root=ARTICLE_ROOT, config=CONFIG)
beta["slot"] = beta["timestamp"].dt.hour * 4 + beta["timestamp"].dt.minute // 15

prediction_columns = [
    "substation_id", "date", "left_slot", "right_slot", "score", "threshold",
    "predicted_day", "true_day", "confidence", "confidence_margin",
    "weight_F1", "weight_F3", "weight_F4",
]
day_predictions = predictions[prediction_columns].copy()
day_predictions = day_predictions.rename(
    columns={"true_day": "prediction_true_day", "confidence": "prediction_confidence"}
)
interval_audit = beta.merge(
    day_predictions,
    on=["substation_id", "date"],
    how="left",
    validate="many_to_one",
)
interval_audit["predicted_interval"] = (
    interval_audit["predicted_day"]
    & interval_audit["slot"].between(
        interval_audit["left_slot"], interval_audit["right_slot"]
    )
)

assert len(interval_audit) == len(beta) == 280_800
assert interval_audit["predicted_day"].notna().all()
assert interval_audit["label_day"].eq(interval_audit["prediction_true_day"]).all()
assert interval_audit["confidence"].eq(interval_audit["prediction_confidence"]).all()
assert predictions.duplicated(["substation_id", "date"]).sum() == 0

INTERVAL_AUDIT_PATH = OUTPUT_DIRS["intermediate"] / "heldout_prediction_audit.parquet"
write_parquet(interval_audit, INTERVAL_AUDIT_PATH)
display(
    pd.Series(
        {
            "interval_rows": len(interval_audit),
            "substation_days": len(predictions),
            "predicted_positive_days": int(predictions["predicted_day"].sum()),
            "predicted_positive_intervals": int(interval_audit["predicted_interval"].sum()),
        },
        name="outer_prediction_audit",
    )
)

## 3. Day And Interval Classification Metrics

Day metrics compare $RPF(d)$ with $\widehat{RPF}(d)$. Interval metrics compare
the manual flag $z(t)$ with the expanded predicted flag $\hat z(t)$. Full-day
interval metrics are primary; 06:00-18:00 is a diagnostic scope.

For either level,

$$
Precision=\frac{TP}{TP+FP},\qquad
Recall=\frac{TP}{TP+FN},\qquad
F1=\frac{2\,Precision\,Recall}{Precision+Recall}.
$$

**Notation**

| Symbol | Meaning |
|---|---|
| $d$ | One Beta substation-day. |
| $t$ | One 15-minute Beta timestamp. |
| $RPF(d)$ | Final manual day label. |
| $\widehat{RPF}(d)$ | Nested outer-fold predicted day label. |
| $z(t)$ | Final manual interval label. |
| $\hat z(t)$ | Predicted interval flag from the selected positive-day window. |
| $TP,FP,FN,TN$ | True-positive, false-positive, false-negative, and true-negative counts. |

In [ ]:
day_metric_parts = []
for confidence_scope, evaluation in [
    ("beta_sure", predictions.loc[predictions["confidence"].eq("sure")]),
    ("beta_all", predictions),
]:
    rows = metric_rows(evaluation)
    rows.insert(0, "confidence_scope", confidence_scope)
    day_metric_parts.append(rows)
day_metrics = pd.concat(day_metric_parts, ignore_index=True)

interval_metric_parts = []
for confidence_scope, confidence_frame in [
    ("beta_sure", interval_audit.loc[interval_audit["confidence"].eq("sure")]),
    ("beta_all", interval_audit),
]:
    for interval_scope, evaluation in [
        ("full_day", confidence_frame),
        ("daytime_06_18", confidence_frame.loc[confidence_frame["slot"].between(24, 72)]),
    ]:
        rows = metric_rows(
            evaluation,
            truth_column="label_interval",
            prediction_column="predicted_interval",
        )
        rows.insert(0, "interval_scope", interval_scope)
        rows.insert(0, "confidence_scope", confidence_scope)
        interval_metric_parts.append(rows)
interval_metrics = pd.concat(interval_metric_parts, ignore_index=True)

display(
    day_metrics.loc[
        day_metrics["aggregation"].isin(["pooled", "macro_substation"]),
        ["confidence_scope", "aggregation", "support", "precision", "recall", "f1"],
    ]
)
display(
    interval_metrics.loc[
        interval_metrics["aggregation"].eq("pooled"),
        [
            "confidence_scope", "interval_scope", "support", "precision", "recall", "f1"
        ],
    ]
)

## 4. Candidate-Window Intersection Over Union

For true interval set $T_d$ and predicted interval set $P_d$,

$$
IoU(d)=\frac{|T_d\cap P_d|}{|T_d\cup P_d|}.
$$

**Notation**

| Symbol | Meaning |
|---|---|
| $T_d$ | Set of manually labelled RPF slots on day $d$. |
| $P_d$ | Set of predicted correction slots on day $d$; empty for a negative decision. |
| $|\cdot|$ | Number of 15-minute slots in a set. |
| $IoU(d)$ | Slot overlap divided by slot union. |

`tp_days_only` includes days where truth and prediction are both positive.
`event_days_truth_or_prediction` additionally includes FP and FN days, assigning
IoU zero when exactly one set is empty. Days with both sets empty are outside
both reported scopes.

In [ ]:
window_rows = []
for (substation, date), group in interval_audit.groupby(
    ["substation_id", "date"], sort=True
):
    true_slots = set(group.loc[group["label_interval"], "slot"].astype(int))
    predicted_slots = set(group.loc[group["predicted_interval"], "slot"].astype(int))
    true_day = bool(group["label_day"].max())
    predicted_day = bool(group["predicted_day"].iloc[0])
    both_windows = bool(true_slots and predicted_slots)
    window_rows.append(
        {
            "substation_id": substation,
            "date": date,
            "confidence": group["confidence"].iloc[0],
            "true_day": true_day,
            "predicted_day": predicted_day,
            "true_interval_count": len(true_slots),
            "predicted_interval_count": len(predicted_slots),
            "window_iou": window_iou(true_slots, predicted_slots),
            "absolute_start_error_minutes": (
                15 * abs(min(predicted_slots) - min(true_slots)) if both_windows else np.nan
            ),
            "absolute_end_error_minutes": (
                15 * abs(max(predicted_slots) - max(true_slots)) if both_windows else np.nan
            ),
        }
    )
window_audit = pd.DataFrame(window_rows)

def summarise_iou(frame, confidence_scope, event_scope, aggregation, substation_id=""):
    values = frame["window_iou"].dropna()
    return {
        "confidence_scope": confidence_scope,
        "event_scope": event_scope,
        "aggregation": aggregation,
        "substation_id": substation_id,
        "support_days": len(values),
        "mean_iou": values.mean() if len(values) else np.nan,
        "median_iou": values.median() if len(values) else np.nan,
        "proportion_iou_at_least_0_50": values.ge(0.50).mean() if len(values) else np.nan,
        "proportion_iou_at_least_0_70": values.ge(0.70).mean() if len(values) else np.nan,
        "median_absolute_start_error_minutes": frame["absolute_start_error_minutes"].median(),
        "median_absolute_end_error_minutes": frame["absolute_end_error_minutes"].median(),
    }

iou_rows = []
for confidence_scope, confidence_frame in [
    ("beta_sure", window_audit.loc[window_audit["confidence"].eq("sure")]),
    ("beta_all", window_audit),
]:
    event_frames = {
        "tp_days_only": confidence_frame.loc[
            confidence_frame["true_day"] & confidence_frame["predicted_day"]
        ],
        "event_days_truth_or_prediction": confidence_frame.loc[
            confidence_frame["true_day"] | confidence_frame["predicted_day"]
        ],
    }
    for event_scope, event_frame in event_frames.items():
        iou_rows.append(
            summarise_iou(event_frame, confidence_scope, event_scope, "pooled")
        )
        for substation, group in event_frame.groupby("substation_id", sort=True):
            iou_rows.append(
                summarise_iou(
                    group, confidence_scope, event_scope, "substation", substation
                )
            )
window_metrics = pd.DataFrame(iou_rows)
display(window_metrics.loc[window_metrics["aggregation"].eq("pooled")])

## 5. Correction-Energy Agreement

At each quarter hour, the energy associated with correcting an incorrectly
positive net-load reading is

$$
e(t)=2\max(y(t),0)\times0.25\ \text{hours}.
$$

Pooled correction-energy IoU is

$$
Energy\ IoU=\frac{\sum_{t:z(t)=1\land\hat z(t)=1}e(t)}
{\sum_{t:z(t)=1\lor\hat z(t)=1}e(t)}.
$$

**Notation**

| Symbol | Meaning |
|---|---|
| $y(t)$ | Observed net load in MW. |
| $e(t)$ | Correction magnitude in MWh for one 15-minute interval. |
| $z(t)$ | Manual interval flag. |
| $\hat z(t)$ | Predicted interval flag. |
| $\land,\lor$ | Logical AND and OR. |

Energy precision, recall, and F1 use the same overlap as their numerator and
predicted/manual correction energy as their denominators. Full day is primary;
daytime is retained as an audit.

In [ ]:
interval_audit["correction_energy_MWh"] = (
    2 * interval_audit["net_load_MW"].clip(lower=0).fillna(0) * 0.25
)
interval_audit["manual_correction_MWh"] = np.where(
    interval_audit["label_interval"], interval_audit["correction_energy_MWh"], 0.0
)
interval_audit["predicted_correction_MWh"] = np.where(
    interval_audit["predicted_interval"], interval_audit["correction_energy_MWh"], 0.0
)
interval_audit["overlap_correction_MWh"] = np.where(
    interval_audit["label_interval"] & interval_audit["predicted_interval"],
    interval_audit["correction_energy_MWh"],
    0.0,
)
interval_audit["union_correction_MWh"] = np.where(
    interval_audit["label_interval"] | interval_audit["predicted_interval"],
    interval_audit["correction_energy_MWh"],
    0.0,
)

def summarise_energy(frame, confidence_scope, interval_scope, aggregation, substation_id=""):
    manual = frame["manual_correction_MWh"].sum()
    predicted = frame["predicted_correction_MWh"].sum()
    overlap = frame["overlap_correction_MWh"].sum()
    union = frame["union_correction_MWh"].sum()
    precision = overlap / predicted if predicted else 0.0
    recall = overlap / manual if manual else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "confidence_scope": confidence_scope,
        "interval_scope": interval_scope,
        "aggregation": aggregation,
        "substation_id": substation_id,
        "manual_correction_MWh": manual,
        "predicted_correction_MWh": predicted,
        "overlap_correction_MWh": overlap,
        "union_correction_MWh": union,
        "energy_precision": precision,
        "energy_recall": recall,
        "energy_f1": f1,
        "energy_iou": overlap / union if union else np.nan,
    }

energy_rows = []
for confidence_scope, confidence_frame in [
    ("beta_sure", interval_audit.loc[interval_audit["confidence"].eq("sure")]),
    ("beta_all", interval_audit),
]:
    for interval_scope, evaluation in [
        ("full_day", confidence_frame),
        ("daytime_06_18", confidence_frame.loc[confidence_frame["slot"].between(24, 72)]),
    ]:
        energy_rows.append(
            summarise_energy(evaluation, confidence_scope, interval_scope, "pooled")
        )
        for substation, group in evaluation.groupby("substation_id", sort=True):
            energy_rows.append(
                summarise_energy(
                    group, confidence_scope, interval_scope, "substation", substation
                )
            )
energy_metrics = pd.DataFrame(energy_rows)
display(energy_metrics.loc[energy_metrics["aggregation"].eq("pooled")])

## 6. Confidence Coverage And Manual Review

For each outer prediction,

$$
margin(d)=|Score(W_d^*)-\tau_d|.
$$

**Notation**

| Symbol | Meaning |
|---|---|
| $Score(W_d^*)$ | Selected candidate's unit-sum weighted physical score. |
| $\tau_d$ | Threshold from the fold that held out day $d$'s substation. |
| $margin(d)$ | Distance from the decision boundary; larger means more confident. |

At each coverage level, the largest-margin positive and negative decisions are
auto-accepted. Remaining days are sent to manual review. Metrics therefore
describe only the accepted subset, while the burden table also records how many
days and true RPF days remain for review.

In [ ]:
sure_predictions = predictions.loc[predictions["confidence"].eq("sure")].copy()
coverage_metrics = confidence_coverage_metrics(
    sure_predictions,
    COVERAGE_CONFIG["levels_pct"],
)
recommended = recommended_coverage(
    coverage_metrics,
    minimum_precision=COVERAGE_CONFIG["minimum_precision"],
    minimum_f1=COVERAGE_CONFIG["minimum_f1"],
)
if recommended is None:
    recommended_table = pd.DataFrame(
        [{
            "qualifying_operating_point_found": False,
            "minimum_precision": COVERAGE_CONFIG["minimum_precision"],
            "minimum_f1": COVERAGE_CONFIG["minimum_f1"],
            "recommended_coverage_pct": np.nan,
            "note": "No tested coverage level satisfies both development constraints.",
        }]
    )
else:
    recommended_table = pd.DataFrame(
        [{
            "qualifying_operating_point_found": True,
            "minimum_precision": COVERAGE_CONFIG["minimum_precision"],
            "minimum_f1": COVERAGE_CONFIG["minimum_f1"],
            "recommended_coverage_pct": recommended["coverage_pct"],
            "precision": recommended["precision"],
            "recall": recommended["recall"],
            "f1": recommended["f1"],
            "manual_review_days": recommended["manual_review_days"],
        }]
    )
display(coverage_metrics)
display(recommended_table)

## 7. Write Metrics, Tables, And Figures

All headline tables are derived from the same nested outer prediction audit.
Figure 4 is the requested dual-axis view: auto-accept coverage on the horizontal
axis, days left for manual review on the left vertical axis, and stacked auto FP
and FN counts on the right vertical axis.

In [ ]:
DAY_PATH = OUTPUT_DIRS["metrics"] / "01_day_metrics.csv"
INTERVAL_PATH = OUTPUT_DIRS["metrics"] / "02_interval_metrics.csv"
WINDOW_PATH = OUTPUT_DIRS["metrics"] / "03_window_iou_metrics.csv"
ENERGY_PATH = OUTPUT_DIRS["metrics"] / "04_energy_metrics.csv"
COVERAGE_PATH = OUTPUT_DIRS["metrics"] / "05_confidence_coverage_metrics.csv"
write_csv(day_metrics, DAY_PATH)
write_csv(interval_metrics, INTERVAL_PATH)
write_csv(window_metrics, WINDOW_PATH)
write_csv(energy_metrics, ENERGY_PATH)
write_csv(coverage_metrics, COVERAGE_PATH)

HEADLINE_PATH = OUTPUT_DIRS["tables"] / "table01_final_headline_metrics.csv"
SUBSTATION_PATH = OUTPUT_DIRS["tables"] / "table02_final_metrics_by_substation.csv"
LOCALISATION_PATH = OUTPUT_DIRS["tables"] / "table03_localisation_and_energy.csv"
RECOMMENDED_PATH = (
    OUTPUT_DIRS["tables"] / "table04_recommended_auto_accept_operating_point.csv"
)
headline = day_metrics.loc[
    day_metrics["aggregation"].isin(["pooled", "macro_substation"])
]
substation_table = day_metrics.loc[day_metrics["aggregation"].eq("substation")]
localisation_table = pd.concat(
    [
        window_metrics.loc[window_metrics["aggregation"].eq("pooled")].assign(
            metric_family="window_iou"
        ),
        energy_metrics.loc[energy_metrics["aggregation"].eq("pooled")].assign(
            metric_family="correction_energy"
        ),
    ],
    ignore_index=True,
    sort=False,
)
write_csv(headline, HEADLINE_PATH)
write_csv(substation_table, SUBSTATION_PATH)
write_csv(localisation_table, LOCALISATION_PATH)
write_csv(recommended_table, RECOMMENDED_PATH)

FIGURE_CONFUSION = OUTPUT_DIRS["figures"] / "fig01_final_confusion_matrices.png"
FIGURE_IOU = OUTPUT_DIRS["figures"] / "fig02_window_iou_distribution.png"
FIGURE_ENERGY = OUTPUT_DIRS["figures"] / "fig03_energy_metric_summary.png"
FIGURE_BURDEN = (
    OUTPUT_DIRS["figures"] / "fig04_auto_accept_manual_review_and_errors.png"
)
FIGURE_COVERAGE = (
    OUTPUT_DIRS["figures"] / "fig05_auto_accept_precision_recall_f1.png"
)
plot_final_confusion_matrices(day_metrics, FIGURE_CONFUSION)
plot_window_iou_distribution(window_audit, FIGURE_IOU)
plot_energy_summary(energy_metrics, FIGURE_ENERGY)
plot_auto_accept_burden(coverage_metrics, FIGURE_BURDEN)
plot_coverage_scores(coverage_metrics, FIGURE_COVERAGE)
display(FIGURE_CONFUSION)
display(FIGURE_IOU)
display(FIGURE_ENERGY)
display(FIGURE_BURDEN)
display(FIGURE_COVERAGE)

## 8. Interpretation And Limitations

Day metrics describe held-out-substation classification. Interval, window, and
energy metrics additionally test whether positive decisions target the correct
part of the curve and recover the material correction magnitude. Confidence
coverage quantifies a possible reduction in manual checking; its recommended
point is a development result, not a deployed safety guarantee.

Beta labels informed earlier feature-family development. Nested LOSO prevents
direct outer-fold leakage but does not make Beta a previously untouched external
dataset. Independent validation is still required before operational use.

In [ ]:
MANIFEST_OUTPUTS = [
    DAY_PATH, INTERVAL_PATH, WINDOW_PATH, ENERGY_PATH, COVERAGE_PATH,
    HEADLINE_PATH, SUBSTATION_PATH, LOCALISATION_PATH, RECOMMENDED_PATH,
    FIGURE_CONFUSION, FIGURE_IOU, FIGURE_ENERGY, FIGURE_BURDEN, FIGURE_COVERAGE,
]
manifest = manifest_payload(
    paths=PATHS,
    config=CONFIG,
    started_at=STARTED_AT,
    inputs=[PATHS.config, PATHS.final_data / "dataset_beta.parquet", PREDICTION_PATH],
    outputs=MANIFEST_OUTPUTS,
    row_counts={
        "beta_intervals": len(interval_audit),
        "beta_substation_days": len(predictions),
        "day_metric_rows": len(day_metrics),
        "interval_metric_rows": len(interval_metrics),
        "coverage_levels": len(coverage_metrics),
    },
)
manifest["local_intermediates"] = [str(INTERVAL_AUDIT_PATH.relative_to(PATHS.article))]
MANIFEST_PATH = write_manifest(PATHS, f"{SLUG}.json", manifest)
inventory = pd.DataFrame({"path": [*MANIFEST_OUTPUTS, INTERVAL_AUDIT_PATH, MANIFEST_PATH]})
inventory["exists"] = inventory["path"].map(Path.exists)
inventory["bytes"] = inventory["path"].map(lambda path: path.stat().st_size)
display(inventory)
assert inventory["exists"].all() and inventory["bytes"].gt(0).all()

## Fast Figure-Only Rerender

Run this cell after the lightweight setup cell whenever only the publication
figures need to change. It reads persisted results, refreshes validated
figure-source caches, and does not repeat candidate generation, fitting, or
evaluation.

In [ ]:
from _cached_figure_rendering import render_notebook_figures

RENDER_ONLY = True
if RENDER_ONLY:
    RENDERED_FIGURES = render_notebook_figures(
        ARTICLE_ROOT,
        '02g_m9_pbm_final_evaluation',
        refresh_sources=True,
    )
    display(pd.Series([str(path) for path in RENDERED_FIGURES], name="rendered_figure"))